# EDA: Player Wages Analysis

Exploratory Data Analysis of FIFA player wages

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats

# Load data
df = pd.read_csv('../data/FIFA23/FIFA23_official_data.csv')
print(f"Data loaded: {df.shape[0]} players")

# Clean position data immediately (remove HTML tags like <span...>)
df['Position'] = df['Position'].str.replace(r'<[^>]+>', '', regex=True).str.strip()

print(df[['ID', 'Name', 'Age', 'Position', 'Overall']].head())

Data loaded: 17660 players
       ID             Name  Age Position  Overall
0  209658      L. Goretzka   27      SUB       87
1  212198  Bruno Fernandes   27      LCM       86
2  224334         M. Acuña   30       LB       85
3  192985     K. De Bruyne   31      RCM       91
4  224232       N. Barella   25      RCM       86


In [ ]:
# Data Cleaning: Parse wages and values
# Wages come as "€115K" or "€190K" — convert to numeric
def parse_currency(val):
    """Convert currency strings like '€115K' to numeric values"""
    if pd.isna(val) or val == '':
        return np.nan
    val = str(val).replace('€', '').strip()
    if 'M' in val:
        return float(val.replace('M', '')) * 1_000_000
    elif 'K' in val:
        return float(val.replace('K', '')) * 1_000
    else:
        return float(val) if val else np.nan

df['wage_numeric'] = df['Wage'].apply(parse_currency)
df['value_numeric'] = df['Value'].apply(parse_currency)

# Position is already cleaned in the first cell
# Keep only real players (overall >= 50, not NaN wage)
df_clean = df[(df['Overall'] >= 50) & (df['wage_numeric'].notna())].copy()

print(f"Cleaned data shape: {df_clean.shape}")
print(f"\nWage statistics:")
print(df_clean['wage_numeric'].describe())
print(f"\nTop 5 earners:")
print(df_clean.nlargest(5, 'wage_numeric')[['Name', 'Club', 'Overall', 'Position', 'wage_numeric']])

Cleaned data shape: (17307, 31)

Wage statistics:
count     17307.000000
mean       8340.662738
std       20656.876463
min           0.000000
25%         550.000000
50%        2000.000000
75%        6000.000000
max      450000.000000
Name: wage_numeric, dtype: float64

Top 5 earners:
               Name             Club  Overall Position  wage_numeric
124      K. Benzema   Real Madrid CF       91       CF      450000.0
41   R. Lewandowski     FC Barcelona       91       ST      420000.0
3      K. De Bruyne  Manchester City       91      RCM      350000.0
125        T. Kroos   Real Madrid CF       88      LCM      310000.0
37         Casemiro   Real Madrid CF       89      CDM      300000.0


In [ ]:
# Plotly interactive visualization: Wages vs Overall Rating
fig = px.scatter(
    df_clean,
    x='Overall',
    y='wage_numeric',
    color='Club',
    hover_data=['Name', 'Club', 'Nationality'],
    opacity=0.6,
    title='Player Wages vs Overall Rating',
    labels={'Overall': 'Overall Rating', 'wage_numeric': 'Weekly Wage (€)'},
    template='plotly_dark'
)

# Add a trend line on top
trend = df_clean.groupby('Overall')['wage_numeric'].median().reset_index()
fig.add_trace(go.Scatter(
    x=trend['Overall'], y=trend['wage_numeric'],
    mode='lines', name='Median wage',
    line=dict(color='white', width=2, dash='dash')
))

fig.update_layout(showlegend=False, height=550)
fig.show()

In [ ]:
# Z-score method — flag players whose wage is far from expected for their rating
df_clean['wage_zscore'] = stats.zscore(np.log1p(df_clean['wage_numeric']))

overpaid   = df_clean[df_clean['wage_zscore'] >  2.5].sort_values('wage_numeric', ascending=False)
underpaid  = df_clean[df_clean['wage_zscore'] < -2.5].sort_values('wage_numeric')

print("=== TOP OVERPAID PLAYERS ===")
print(overpaid[['Name', 'Club', 'Position', 'Overall', 'wage_numeric']].head(10))

print("\n=== UNDERPAID GEMS ===")
print(underpaid[['Name', 'Club', 'Position', 'Overall', 'wage_numeric']].head(10))

=== TOP OVERPAID PLAYERS ===
               Name             Club Position  Overall  wage_numeric
124      K. Benzema   Real Madrid CF       CF       91      450000.0
41   R. Lewandowski     FC Barcelona       ST       91      420000.0
3      K. De Bruyne  Manchester City      RCM       91      350000.0
125        T. Kroos   Real Madrid CF      LCM       88      310000.0
37         Casemiro   Real Madrid CF      CDM       89      300000.0
25         M. Salah        Liverpool       RW       90      270000.0
66   Bernardo Silva  Manchester City      LCM       88      260000.0
378    22 S. Agüero     FC Barcelona      SUB       87      260000.0
9      João Cancelo  Manchester City       LB       88      250000.0
694      A. Rüdiger   Real Madrid CF      RCB       87      250000.0

=== UNDERPAID GEMS ===
                  Name Club Position  Overall  wage_numeric
162   21 Santi Cazorla  NaN      SUB       82           0.0
383        21 M. Nérez  NaN       LB       80           0.0
435     